In [1]:

"""
CN7 07 - Nested Feature Selection + Logistic Tuning
===================================================

핵심 목적
---------
CN7_06의 top10_side는 전체 데이터의 label을 보고 EDA에서 선택된 변수이므로,
그 상태로 CV에 넣으면 feature-selection leakage가 생길 수 있다.

이번 단계에서는 각 validation fold마다:
    1) TRAIN 데이터만 사용해 feature ranking
    2) TRAIN에서 Top-K 선택
    3) scaler / C / positive class weight 튜닝
    4) validation 예측

을 수행한다.

평가 우선순위
-------------
1) PR-AUC
2) Recall >= 80%에서 Precision
3) Recall >= 90%에서 Precision
4) Top 5% Precision / Recall

검증
----
A. Repeated StratifiedGroupKFold
B. Purged Time-Block CV

금지
----
- Reason
- TimeStamp
- PART_FACT_PLAN_DATE
- PART_FACT_SERIAL
- validation label을 이용한 feature selection
"""

from pathlib import Path
from itertools import product
import warnings
import json
import numpy as np
import pandas as pd

from scipy.stats import ks_2samp

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

# =============================================================================
# 0. 설정
# =============================================================================

CN7_WORK_DIR = Path(
    r"C:\Users\Administrator\Desktop\kamp\다시하는 cn7"
)

RAW_PATH = (
    CN7_WORK_DIR
    / "cn7_03_eda"
    / "cn7_train_raw_1211.csv"
)

OUT_DIR = (
    CN7_WORK_DIR
    / "cn7_07_nested_tuning"
)
OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TARGET = "PassOrFail"

SENSOR_COLS = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Clamp_Close_Time",
    "Cushion_Position",
    "Plasticizing_Position",
    "Clamp_Open_Position",
    "Max_Injection_Speed",
    "Max_Screw_RPM",
    "Average_Screw_RPM",
    "Max_Injection_Pressure",
    "Max_Switch_Over_Pressure",
    "Max_Back_Pressure",
    "Average_Back_Pressure",
    "Barrel_Temperature_1",
    "Barrel_Temperature_2",
    "Barrel_Temperature_3",
    "Barrel_Temperature_4",
    "Barrel_Temperature_5",
    "Barrel_Temperature_6",
    "Hopper_Temperature",
    "Mold_Temperature_3",
    "Mold_Temperature_4",
]

# -------------------------------------------------------------------------
# 작은 grid만 사용
# 너무 큰 search로 17개 불량에 과적합하지 않도록 제한
# -------------------------------------------------------------------------

TOP_K_VALUES = [
    5,
    8,
    10,
    15,
    23,
]

SCALERS = [
    "standard",
    "robust",
]

C_VALUES = [
    0.03,
    0.10,
    0.30,
]

POS_WEIGHTS = [
    20,
    70,
]

USE_SIDE_VALUES = [
    False,
    True,
]

# 총 5 * 2 * 3 * 2 * 2 = 120 configs

GROUP_SEEDS = [
    42,
    52,
    62,
    72,
    82,
]

N_SPLITS = 5

TIME_BLOCK_MINUTES = 10
PURGE_MINUTES = 10

RECALL_TARGETS = [
    0.80,
    0.90,
]


# =============================================================================
# 유틸
# =============================================================================

def banner(title):
    print(
        "\n"
        + "=" * 120
    )
    print(title)
    print(
        "=" * 120
    )


def make_config_id(
    top_k,
    scaler,
    c_value,
    pos_weight,
    use_side,
):
    return (
        f"k{top_k}"
        f"__{scaler}"
        f"__C{c_value:g}"
        f"__W{pos_weight}"
        f"__side{int(use_side)}"
    )


def effect_shift_score(
    normal,
    defect,
):
    """
    tiny positive sample에서도 동작하도록
    정상군 std를 기준으로 평균 이동량 계산.
    """
    normal = np.asarray(
        normal,
        dtype=float,
    )

    defect = np.asarray(
        defect,
        dtype=float,
    )

    if (
        len(normal) == 0
        or len(defect) == 0
    ):
        return 0.0

    sd = np.nanstd(
        normal,
        ddof=0,
    )

    if (
        not np.isfinite(sd)
        or sd < 1e-12
    ):
        return 0.0

    return float(
        abs(
            np.nanmean(defect)
            - np.nanmean(normal)
        )
        / sd
    )


def rank_features_train(
    df_train,
    feature_cols,
    target_col,
):
    """
    TRAIN 데이터만 이용해 feature ranking.

    score =
        KS statistic
        + 0.5 * clipped standardized mean shift

    p-value는 ranking에 사용하지 않는다.
    """
    y = (
        df_train[target_col]
        .astype(int)
    )

    rows = []

    for c in feature_cols:

        x0 = pd.to_numeric(
            df_train.loc[
                y == 0,
                c,
            ],
            errors="coerce",
        ).dropna()

        x1 = pd.to_numeric(
            df_train.loc[
                y == 1,
                c,
            ],
            errors="coerce",
        ).dropna()

        if (
            len(x0) == 0
            or len(x1) == 0
        ):
            ks_stat = 0.0
            shift = 0.0

        else:
            try:
                ks_stat = float(
                    ks_2samp(
                        x0,
                        x1,
                    ).statistic
                )
            except Exception:
                ks_stat = 0.0

            shift = effect_shift_score(
                x0,
                x1,
            )

        # 너무 큰 outlier 효과가 ranking을 독점하지 않도록 cap
        shift_clipped = min(
            shift,
            4.0,
        )

        score = (
            ks_stat
            + 0.5 * shift_clipped
        )

        rows.append({
            "feature": c,
            "KS_stat": ks_stat,
            "mean_shift_normal_sd": shift,
            "selection_score": score,
        })

    rank_df = (
        pd.DataFrame(rows)
        .sort_values(
            [
                "selection_score",
                "KS_stat",
                "feature",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    rank_df[
        "rank"
    ] = (
        np.arange(
            len(rank_df)
        )
        + 1
    )

    return rank_df


def make_model(
    scaler_name,
    c_value,
    pos_weight,
    random_state,
):
    if scaler_name == "standard":
        scaler = StandardScaler()

    elif scaler_name == "robust":
        scaler = RobustScaler()

    else:
        raise ValueError(
            scaler_name
        )

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
            ),
        ),
        (
            "scaler",
            scaler,
        ),
        (
            "model",
            LogisticRegression(
                C=c_value,
                penalty="l2",
                class_weight={
                    0: 1.0,
                    1: float(
                        pos_weight
                    ),
                },
                max_iter=5000,
                solver="liblinear",
                random_state=random_state,
            ),
        ),
    ])


def score_predictions(
    y_true,
    score,
):
    y = np.asarray(
        y_true
    ).astype(int)

    s = np.asarray(
        score
    ).astype(float)

    valid = np.isfinite(
        s
    )

    y = y[valid]
    s = s[valid]

    if (
        len(y) == 0
        or len(
            np.unique(y)
        ) < 2
    ):
        return {
            "n": len(y),
            "defects": int(
                y.sum()
            ),
            "PR_AUC": np.nan,
            "ROC_AUC": np.nan,
        }

    return {
        "n": len(y),
        "defects": int(
            y.sum()
        ),
        "PR_AUC": float(
            average_precision_score(
                y,
                s,
            )
        ),
        "ROC_AUC": float(
            roc_auc_score(
                y,
                s,
            )
        ),
    }


def threshold_table(
    y_true,
    score,
):
    y = np.asarray(
        y_true
    ).astype(int)

    s = np.asarray(
        score
    ).astype(float)

    valid = np.isfinite(
        s
    )

    y = y[valid]
    s = s[valid]

    thresholds = np.sort(
        np.unique(s)
    )[::-1]

    rows = []

    for t in thresholds:

        pred = (
            s >= t
        ).astype(int)

        tn, fp, fn, tp = (
            confusion_matrix(
                y,
                pred,
                labels=[0, 1],
            ).ravel()
        )

        precision = (
            tp / (tp + fp)
            if (
                tp + fp
            ) > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if (
                tp + fn
            ) > 0
            else 0.0
        )

        rows.append({
            "threshold": float(t),
            "precision": float(
                precision
            ),
            "recall": float(
                recall
            ),
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn),
            "predicted_positive": int(
                tp + fp
            ),
        })

    return pd.DataFrame(
        rows
    )


def best_precision_at_recall(
    y_true,
    score,
    recall_target,
):
    tab = threshold_table(
        y_true,
        score,
    )

    cand = tab[
        tab["recall"]
        >= recall_target
    ].copy()

    if len(cand) == 0:
        return {
            "threshold": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "TP": np.nan,
            "FP": np.nan,
            "FN": np.nan,
        }

    cand = cand.sort_values(
        [
            "precision",
            "recall",
            "predicted_positive",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )

    r = cand.iloc[0]

    return {
        "threshold": float(
            r["threshold"]
        ),
        "precision": float(
            r["precision"]
        ),
        "recall": float(
            r["recall"]
        ),
        "TP": int(
            r["TP"]
        ),
        "FP": int(
            r["FP"]
        ),
        "FN": int(
            r["FN"]
        ),
    }


def top_fraction_metric(
    y_true,
    score,
    frac=0.05,
):
    y = np.asarray(
        y_true
    ).astype(int)

    s = np.asarray(
        score
    ).astype(float)

    valid = np.isfinite(
        s
    )

    y = y[valid]
    s = s[valid]

    k = max(
        1,
        int(
            np.ceil(
                len(y) * frac
            )
        ),
    )

    order = np.argsort(
        -s
    )

    idx = order[:k]

    tp = int(
        y[idx].sum()
    )

    fp = int(
        k - tp
    )

    fn = int(
        y.sum()
        - tp
    )

    return {
        "k": k,
        "precision": float(
            tp / k
        ),
        "recall": float(
            tp / y.sum()
            if y.sum() > 0
            else np.nan
        ),
        "TP": tp,
        "FP": fp,
        "FN": fn,
    }


# =============================================================================
# 1. 데이터 로드
# =============================================================================

banner(
    "CN7 07 - Nested Feature Selection + Logistic Tuning"
)

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"없음: {RAW_PATH}"
    )

df = pd.read_csv(
    RAW_PATH
)

df[TARGET] = pd.to_numeric(
    df[TARGET],
    errors="raise",
).astype(int)

if "TimeStamp_dt" in df.columns:

    df[
        "TimeStamp_dt"
    ] = pd.to_datetime(
        df[
            "TimeStamp_dt"
        ],
        errors="coerce",
    )

else:

    df[
        "TimeStamp_dt"
    ] = pd.to_datetime(
        df["TimeStamp"],
        errors="coerce",
    )

if "Side" not in df.columns:

    s = (
        df["PART_NAME"]
        .astype(str)
        .str.upper()
    )

    df[
        "Side"
    ] = np.where(
        s.str.contains(
            r"\bRH\b",
            regex=True,
            na=False,
        ),
        "RH",
        "LH",
    )

df[
    "Side_RH"
] = (
    df["Side"] == "RH"
).astype(int)

constant_cols = [
    c
    for c in SENSOR_COLS
    if (
        c in df.columns
        and df[c].nunique(
            dropna=False
        ) <= 1
    )
]

base_features = [
    c
    for c in SENSOR_COLS
    if (
        c in df.columns
        and c not in constant_cols
    )
]

print(
    "shape:",
    df.shape,
)

print(
    "defects:",
    int(
        df[TARGET].sum()
    ),
    "/",
    len(df),
)

print(
    "features:",
    len(
        base_features
    ),
)

print(
    "removed constant:",
    constant_cols,
)


# =============================================================================
# 2. Group 생성
# =============================================================================

banner(
    "1) Sensor Group"
)

df[
    "sensor_group_id"
] = (
    df.groupby(
        base_features,
        dropna=False,
        sort=False,
    )
    .ngroup()
)

groups = df[
    "sensor_group_id"
]

y = df[
    TARGET
]

print(
    "groups:",
    groups.nunique(),
)

print(
    "positive groups:",
    int(
        df.groupby(
            "sensor_group_id"
        )[TARGET]
        .max()
        .sum()
    ),
)


# =============================================================================
# 3. Config 생성
# =============================================================================

banner(
    "2) Hyperparameter Grid"
)

configs = []

for (
    top_k,
    scaler,
    c_value,
    pos_weight,
    use_side,
) in product(
    TOP_K_VALUES,
    SCALERS,
    C_VALUES,
    POS_WEIGHTS,
    USE_SIDE_VALUES,
):

    configs.append({
        "config_id":
            make_config_id(
                top_k,
                scaler,
                c_value,
                pos_weight,
                use_side,
            ),
        "top_k": top_k,
        "scaler": scaler,
        "C": c_value,
        "pos_weight": pos_weight,
        "use_side": use_side,
    })

print(
    "configs:",
    len(configs),
)


# =============================================================================
# 4. Group CV fold 사전 생성 + train-only ranking
# =============================================================================

banner(
    "3) Group CV fold별 train-only feature ranking"
)

group_folds = []
group_rank_rows = []

fold_global_id = 0

for repeat_no, seed in enumerate(
    GROUP_SEEDS,
    start=1,
):

    cv = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed,
    )

    for fold_no, (
        tr_idx,
        va_idx,
    ) in enumerate(
        cv.split(
            df,
            y,
            groups,
        ),
        start=1,
    ):

        fold_global_id += 1

        rank_df = (
            rank_features_train(
                df.iloc[
                    tr_idx
                ],
                base_features,
                TARGET,
            )
        )

        ranked = rank_df[
            "feature"
        ].tolist()

        group_folds.append({
            "fold_global_id":
                fold_global_id,
            "repeat":
                repeat_no,
            "fold":
                fold_no,
            "seed":
                seed,
            "train_idx":
                tr_idx,
            "valid_idx":
                va_idx,
            "ranked_features":
                ranked,
        })

        tmp = (
            rank_df.copy()
        )

        tmp[
            "fold_global_id"
        ] = fold_global_id

        tmp[
            "repeat"
        ] = repeat_no

        tmp[
            "fold"
        ] = fold_no

        group_rank_rows.append(
            tmp
        )

        print(
            f"repeat={repeat_no}, "
            f"fold={fold_no}, "
            f"train_pos={int(y.iloc[tr_idx].sum())}, "
            f"valid_pos={int(y.iloc[va_idx].sum())}, "
            f"top5={ranked[:5]}"
        )

group_rank_df = pd.concat(
    group_rank_rows,
    ignore_index=True,
)

group_rank_df.to_csv(
    OUT_DIR
    / "group_fold_feature_rankings.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 5. Feature ranking 안정성
# =============================================================================

banner(
    "4) Feature ranking 안정성"
)

ranking_stability = (
    group_rank_df
    .groupby(
        "feature"
    )
    .agg(
        mean_rank=(
            "rank",
            "mean",
        ),
        median_rank=(
            "rank",
            "median",
        ),
        rank_std=(
            "rank",
            "std",
        ),
        top5_rate=(
            "rank",
            lambda x:
                float(
                    (x <= 5).mean()
                ),
        ),
        top10_rate=(
            "rank",
            lambda x:
                float(
                    (x <= 10).mean()
                ),
        ),
    )
    .reset_index()
    .sort_values(
        [
            "mean_rank",
            "top10_rate",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

print(
    ranking_stability
    .round(3)
    .to_string(
        index=False
    )
)

ranking_stability.to_csv(
    OUT_DIR
    / "feature_ranking_stability.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 6. Repeated Group CV Tuning
# =============================================================================

banner(
    "5) Repeated Group CV tuning"
)

group_results = []
group_oof_rows = []

for config_no, config in enumerate(
    configs,
    start=1,
):

    pred_sum = np.zeros(
        len(df),
        dtype=float,
    )

    pred_count = np.zeros(
        len(df),
        dtype=int,
    )

    fold_pr = []

    for fold_info in group_folds:

        tr_idx = fold_info[
            "train_idx"
        ]

        va_idx = fold_info[
            "valid_idx"
        ]

        selected = (
            fold_info[
                "ranked_features"
            ][
                : config[
                    "top_k"
                ]
            ]
        )

        use_cols = list(
            selected
        )

        if config[
            "use_side"
        ]:
            use_cols.append(
                "Side_RH"
            )

        X_train = (
            df.iloc[
                tr_idx
            ][
                use_cols
            ]
            .astype(float)
        )

        X_valid = (
            df.iloc[
                va_idx
            ][
                use_cols
            ]
            .astype(float)
        )

        model = make_model(
            scaler_name=
                config[
                    "scaler"
                ],
            c_value=
                config[
                    "C"
                ],
            pos_weight=
                config[
                    "pos_weight"
                ],
            random_state=
                fold_info[
                    "seed"
                ]
                + fold_info[
                    "fold"
                ],
        )

        model.fit(
            X_train,
            y.iloc[
                tr_idx
            ],
        )

        pred = (
            model.predict_proba(
                X_valid
            )[:, 1]
        )

        pred_sum[
            va_idx
        ] += pred

        pred_count[
            va_idx
        ] += 1

        if (
            y.iloc[
                va_idx
            ].nunique()
            == 2
        ):

            fold_pr.append(
                average_precision_score(
                    y.iloc[
                        va_idx
                    ],
                    pred,
                )
            )

    oof = np.divide(
        pred_sum,
        pred_count,
        out=np.full(
            len(df),
            np.nan,
        ),
        where=(
            pred_count > 0
        ),
    )

    metric = score_predictions(
        y,
        oof,
    )

    op80 = (
        best_precision_at_recall(
            y,
            oof,
            0.80,
        )
    )

    op90 = (
        best_precision_at_recall(
            y,
            oof,
            0.90,
        )
    )

    top5 = top_fraction_metric(
        y,
        oof,
        frac=0.05,
    )

    row = {
        **config,
        "cv":
            "repeated_group",
        "PR_AUC":
            metric[
                "PR_AUC"
            ],
        "ROC_AUC_ref":
            metric[
                "ROC_AUC"
            ],
        "R80_precision":
            op80[
                "precision"
            ],
        "R80_recall":
            op80[
                "recall"
            ],
        "R80_threshold":
            op80[
                "threshold"
            ],
        "R80_TP":
            op80[
                "TP"
            ],
        "R80_FP":
            op80[
                "FP"
            ],
        "R90_precision":
            op90[
                "precision"
            ],
        "R90_recall":
            op90[
                "recall"
            ],
        "R90_threshold":
            op90[
                "threshold"
            ],
        "R90_TP":
            op90[
                "TP"
            ],
        "R90_FP":
            op90[
                "FP"
            ],
        "Top5_precision":
            top5[
                "precision"
            ],
        "Top5_recall":
            top5[
                "recall"
            ],
        "fold_PR_mean":
            float(
                np.mean(
                    fold_pr
                )
            )
            if fold_pr
            else np.nan,
        "fold_PR_std":
            float(
                np.std(
                    fold_pr,
                    ddof=1,
                )
            )
            if len(
                fold_pr
            ) > 1
            else np.nan,
    }

    group_results.append(
        row
    )

    for i in range(
        len(df)
    ):

        group_oof_rows.append({
            "config_id":
                config[
                    "config_id"
                ],
            "row_index":
                i,
            "target":
                int(
                    y.iloc[i]
                ),
            "prediction":
                oof[i],
            "sensor_group_id":
                int(
                    groups.iloc[i]
                ),
        })

    if (
        config_no % 10 == 0
        or config_no == 1
        or config_no == len(
            configs
        )
    ):

        print(
            f"[{config_no:3d}/"
            f"{len(configs)}] "
            f"{config['config_id']} | "
            f"PR={row['PR_AUC']:.4f} | "
            f"R80P={row['R80_precision']:.3f} | "
            f"R90P={row['R90_precision']:.3f}"
        )

group_result_df = (
    pd.DataFrame(
        group_results
    )
)

group_result_df.to_csv(
    OUT_DIR
    / "group_tuning_results.csv",
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame(
    group_oof_rows
).to_csv(
    OUT_DIR
    / "group_oof_all_configs.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 7. Purged fold 사전 생성 + train-only ranking
# =============================================================================

banner(
    "6) Purged fold별 train-only feature ranking"
)

df[
    "time_block"
] = (
    df[
        "TimeStamp_dt"
    ]
    .dt.floor(
        f"{TIME_BLOCK_MINUTES}min"
    )
)

unique_blocks = sorted(
    df[
        "time_block"
    ]
    .dropna()
    .unique()
)

purged_folds = []
purged_rank_rows = []

purged_id = 0

for block in unique_blocks:

    block = pd.Timestamp(
        block
    )

    val_mask = (
        df[
            "time_block"
        ] == block
    )

    va_idx = np.where(
        val_mask.to_numpy()
    )[0]

    if len(
        va_idx
    ) == 0:
        continue

    block_start = block

    block_end = (
        block
        + pd.Timedelta(
            minutes=
                TIME_BLOCK_MINUTES
        )
    )

    purge_start = (
        block_start
        - pd.Timedelta(
            minutes=
                PURGE_MINUTES
        )
    )

    purge_end = (
        block_end
        + pd.Timedelta(
            minutes=
                PURGE_MINUTES
        )
    )

    train_mask = ~(
        (
            df[
                "TimeStamp_dt"
            ] >= purge_start
        )
        &
        (
            df[
                "TimeStamp_dt"
            ] < purge_end
        )
    )

    val_groups = set(
        df.loc[
            val_mask,
            "sensor_group_id",
        ].tolist()
    )

    train_mask &= ~df[
        "sensor_group_id"
    ].isin(
        val_groups
    )

    tr_idx = np.where(
        train_mask.to_numpy()
    )[0]

    if (
        len(tr_idx) == 0
        or y.iloc[
            tr_idx
        ].nunique() < 2
    ):
        continue

    purged_id += 1

    rank_df = (
        rank_features_train(
            df.iloc[
                tr_idx
            ],
            base_features,
            TARGET,
        )
    )

    ranked = rank_df[
        "feature"
    ].tolist()

    purged_folds.append({
        "purged_id":
            purged_id,
        "block":
            block,
        "train_idx":
            tr_idx,
        "valid_idx":
            va_idx,
        "ranked_features":
            ranked,
    })

    tmp = (
        rank_df.copy()
    )

    tmp[
        "purged_id"
    ] = purged_id

    tmp[
        "block"
    ] = block

    purged_rank_rows.append(
        tmp
    )

print(
    "usable purged folds:",
    len(
        purged_folds
    ),
)

if purged_rank_rows:

    pd.concat(
        purged_rank_rows,
        ignore_index=True,
    ).to_csv(
        OUT_DIR
        / "purged_fold_feature_rankings.csv",
        index=False,
        encoding="utf-8-sig",
    )


# =============================================================================
# 8. Purged tuning
# =============================================================================

banner(
    "7) Purged Time-Block tuning"
)

purged_results = []
purged_oof_rows = []

for config_no, config in enumerate(
    configs,
    start=1,
):

    oof = np.full(
        len(df),
        np.nan,
        dtype=float,
    )

    for fold_info in purged_folds:

        tr_idx = fold_info[
            "train_idx"
        ]

        va_idx = fold_info[
            "valid_idx"
        ]

        selected = (
            fold_info[
                "ranked_features"
            ][
                : config[
                    "top_k"
                ]
            ]
        )

        use_cols = list(
            selected
        )

        if config[
            "use_side"
        ]:
            use_cols.append(
                "Side_RH"
            )

        model = make_model(
            scaler_name=
                config[
                    "scaler"
                ],
            c_value=
                config[
                    "C"
                ],
            pos_weight=
                config[
                    "pos_weight"
                ],
            random_state=
                1000
                + fold_info[
                    "purged_id"
                ],
        )

        model.fit(
            df.iloc[
                tr_idx
            ][
                use_cols
            ].astype(float),
            y.iloc[
                tr_idx
            ],
        )

        pred = (
            model.predict_proba(
                df.iloc[
                    va_idx
                ][
                    use_cols
                ].astype(float)
            )[:, 1]
        )

        oof[
            va_idx
        ] = pred

    metric = score_predictions(
        y,
        oof,
    )

    op80 = (
        best_precision_at_recall(
            y,
            oof,
            0.80,
        )
    )

    op90 = (
        best_precision_at_recall(
            y,
            oof,
            0.90,
        )
    )

    top5 = top_fraction_metric(
        y,
        oof,
        frac=0.05,
    )

    row = {
        **config,
        "cv":
            "purged_time_block",
        "PR_AUC":
            metric[
                "PR_AUC"
            ],
        "ROC_AUC_ref":
            metric[
                "ROC_AUC"
            ],
        "R80_precision":
            op80[
                "precision"
            ],
        "R80_recall":
            op80[
                "recall"
            ],
        "R80_threshold":
            op80[
                "threshold"
            ],
        "R80_TP":
            op80[
                "TP"
            ],
        "R80_FP":
            op80[
                "FP"
            ],
        "R90_precision":
            op90[
                "precision"
            ],
        "R90_recall":
            op90[
                "recall"
            ],
        "R90_threshold":
            op90[
                "threshold"
            ],
        "R90_TP":
            op90[
                "TP"
            ],
        "R90_FP":
            op90[
                "FP"
            ],
        "Top5_precision":
            top5[
                "precision"
            ],
        "Top5_recall":
            top5[
                "recall"
            ],
    }

    purged_results.append(
        row
    )

    for i in range(
        len(df)
    ):

        purged_oof_rows.append({
            "config_id":
                config[
                    "config_id"
                ],
            "row_index":
                i,
            "target":
                int(
                    y.iloc[i]
                ),
            "prediction":
                oof[i],
            "sensor_group_id":
                int(
                    groups.iloc[i]
                ),
        })

    if (
        config_no % 10 == 0
        or config_no == 1
        or config_no == len(
            configs
        )
    ):

        print(
            f"[{config_no:3d}/"
            f"{len(configs)}] "
            f"{config['config_id']} | "
            f"PR={row['PR_AUC']:.4f} | "
            f"R80P={row['R80_precision']:.3f} | "
            f"R90P={row['R90_precision']:.3f}"
        )

purged_result_df = (
    pd.DataFrame(
        purged_results
    )
)

purged_result_df.to_csv(
    OUT_DIR
    / "purged_tuning_results.csv",
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame(
    purged_oof_rows
).to_csv(
    OUT_DIR
    / "purged_oof_all_configs.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 9. 결합
# =============================================================================

banner(
    "8) Group + Purged 통합 순위"
)

merged = (
    group_result_df
    .merge(
        purged_result_df,
        on=[
            "config_id",
            "top_k",
            "scaler",
            "C",
            "pos_weight",
            "use_side",
        ],
        suffixes=(
            "_group",
            "_purged",
        ),
    )
)

merged[
    "PR_min"
] = merged[
    [
        "PR_AUC_group",
        "PR_AUC_purged",
    ]
].min(
    axis=1
)

merged[
    "PR_mean"
] = merged[
    [
        "PR_AUC_group",
        "PR_AUC_purged",
    ]
].mean(
    axis=1
)

merged[
    "R80P_min"
] = merged[
    [
        "R80_precision_group",
        "R80_precision_purged",
    ]
].min(
    axis=1
)

merged[
    "R90P_min"
] = merged[
    [
        "R90_precision_group",
        "R90_precision_purged",
    ]
].min(
    axis=1
)

merged = merged.sort_values(
    [
        "PR_min",
        "PR_mean",
        "R80P_min",
        "R90P_min",
    ],
    ascending=False,
)

merged.to_csv(
    OUT_DIR
    / "nested_tuning_combined.csv",
    index=False,
    encoding="utf-8-sig",
)

show_cols = [
    "config_id",
    "top_k",
    "scaler",
    "C",
    "pos_weight",
    "use_side",
    "PR_AUC_group",
    "PR_AUC_purged",
    "PR_min",
    "R80_precision_group",
    "R80_recall_group",
    "R80_precision_purged",
    "R80_recall_purged",
    "R90_precision_group",
    "R90_recall_group",
    "R90_precision_purged",
    "R90_recall_purged",
    "Top5_precision_group",
    "Top5_recall_group",
]

print(
    merged[
        show_cols
    ]
    .head(30)
    .round(4)
    .to_string(
        index=False
    )
)


# =============================================================================
# 10. Group PR 상위
# =============================================================================

banner(
    "9) Group PR-AUC 상위 20"
)

group_top = (
    group_result_df
    .sort_values(
        [
            "PR_AUC",
            "R80_precision",
        ],
        ascending=[
            False,
            False,
        ],
    )
)

print(
    group_top[
        [
            "config_id",
            "top_k",
            "scaler",
            "C",
            "pos_weight",
            "use_side",
            "PR_AUC",
            "R80_precision",
            "R80_recall",
            "R90_precision",
            "R90_recall",
            "Top5_precision",
            "Top5_recall",
            "fold_PR_mean",
            "fold_PR_std",
        ]
    ]
    .head(20)
    .round(4)
    .to_string(
        index=False
    )
)


# =============================================================================
# 11. Purged PR 상위
# =============================================================================

banner(
    "10) Purged PR-AUC 상위 20"
)

purged_top = (
    purged_result_df
    .sort_values(
        [
            "PR_AUC",
            "R80_precision",
        ],
        ascending=[
            False,
            False,
        ],
    )
)

print(
    purged_top[
        [
            "config_id",
            "top_k",
            "scaler",
            "C",
            "pos_weight",
            "use_side",
            "PR_AUC",
            "R80_precision",
            "R80_recall",
            "R90_precision",
            "R90_recall",
            "Top5_precision",
            "Top5_recall",
        ]
    ]
    .head(20)
    .round(4)
    .to_string(
        index=False
    )
)


# =============================================================================
# 12. 최종 shortlist
# =============================================================================

banner(
    "11) Shortlist"
)

shortlist = []

# Group 1등
r = group_top.iloc[0]

shortlist.append({
    "criterion":
        "best_group_PR",
    "config_id":
        r["config_id"],
    "PR_group":
        r["PR_AUC"],
    "R80P_group":
        r["R80_precision"],
    "R90P_group":
        r["R90_precision"],
})

# Purged 1등
r = purged_top.iloc[0]

shortlist.append({
    "criterion":
        "best_purged_PR",
    "config_id":
        r["config_id"],
    "PR_purged":
        r["PR_AUC"],
    "R80P_purged":
        r["R80_precision"],
    "R90P_purged":
        r["R90_precision"],
})

# robustness
r = merged.iloc[0]

shortlist.append({
    "criterion":
        "best_PR_min",
    "config_id":
        r["config_id"],
    "PR_group":
        r["PR_AUC_group"],
    "PR_purged":
        r["PR_AUC_purged"],
    "PR_min":
        r["PR_min"],
    "R80P_group":
        r["R80_precision_group"],
    "R80P_purged":
        r["R80_precision_purged"],
})

shortlist_df = (
    pd.DataFrame(
        shortlist
    )
)

print(
    shortlist_df
    .round(4)
    .to_string(
        index=False
    )
)

shortlist_df.to_csv(
    OUT_DIR
    / "shortlist.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 13. 자동 요약
# =============================================================================

banner(
    "12) 자동 요약"
)

best_group = (
    group_top.iloc[0]
)

best_purged = (
    purged_top.iloc[0]
)

best_robust = (
    merged.iloc[0]
)

lines = []

lines.append(
    f"- Nested Group 최고: "
    f"{best_group['config_id']} | "
    f"PR={best_group['PR_AUC']:.4f}, "
    f"R80 Precision={best_group['R80_precision']:.4f}, "
    f"R80 Recall={best_group['R80_recall']:.4f}, "
    f"R90 Precision={best_group['R90_precision']:.4f}"
)

lines.append(
    f"- Nested Purged 최고: "
    f"{best_purged['config_id']} | "
    f"PR={best_purged['PR_AUC']:.4f}, "
    f"R80 Precision={best_purged['R80_precision']:.4f}, "
    f"R80 Recall={best_purged['R80_recall']:.4f}, "
    f"R90 Precision={best_purged['R90_precision']:.4f}"
)

lines.append(
    f"- PR_min 최고: "
    f"{best_robust['config_id']} | "
    f"Group={best_robust['PR_AUC_group']:.4f}, "
    f"Purged={best_robust['PR_AUC_purged']:.4f}"
)

lines.append(
    "- 이번 결과는 feature selection이 fold 내부에서 수행되므로 "
    "CN7_06의 fixed Top10 결과보다 validation 관점에서 더 신뢰할 수 있다."
)

lines.append(
    "- 다음 단계는 shortlist 2~4개만 이용해 probability averaging ensemble을 검증한다."
)

for line in lines:
    print(
        line
    )

with open(
    OUT_DIR
    / "nested_tuning_summary.txt",
    "w",
    encoding="utf-8",
) as f:

    f.write(
        "\n".join(
            lines
        )
    )


# =============================================================================
# 완료
# =============================================================================

banner(
    "완료"
)

print(
    "결과 폴더:",
    OUT_DIR,
)

print()

print(
    "핵심 파일:"
)

print(
    "  1) feature_ranking_stability.csv"
)

print(
    "  2) group_tuning_results.csv"
)

print(
    "  3) purged_tuning_results.csv"
)

print(
    "  4) nested_tuning_combined.csv"
)

print(
    "  5) shortlist.csv"
)

print(
    "  6) nested_tuning_summary.txt"
)

print()

print(
    "실행 후 8) Group + Purged 통합 순위부터 "
    "12) 자동 요약까지 보내주세요."
)



CN7 07 - Nested Feature Selection + Logistic Tuning
shape: (1211, 52)
defects: 17 / 1211
features: 23
removed constant: ['Clamp_Open_Position']

1) Sensor Group
groups: 606
positive groups: 14

2) Hyperparameter Grid
configs: 120

3) Group CV fold별 train-only feature ranking
repeat=1, fold=1, train_pos=13, valid_pos=4, top5=['Max_Injection_Speed', 'Plasticizing_Position', 'Max_Back_Pressure', 'Mold_Temperature_4', 'Mold_Temperature_3']
repeat=1, fold=2, train_pos=14, valid_pos=3, top5=['Max_Injection_Speed', 'Max_Back_Pressure', 'Plasticizing_Position', 'Average_Back_Pressure', 'Mold_Temperature_4']
repeat=1, fold=3, train_pos=14, valid_pos=3, top5=['Max_Injection_Speed', 'Plasticizing_Position', 'Max_Back_Pressure', 'Mold_Temperature_4', 'Mold_Temperature_3']
repeat=1, fold=4, train_pos=13, valid_pos=4, top5=['Max_Injection_Speed', 'Plasticizing_Position', 'Max_Back_Pressure', 'Mold_Temperature_4', 'Mold_Temperature_3']
repeat=1, fold=5, train_pos=14, valid_pos=3, top5=['Max_Injectio